<a href="https://colab.research.google.com/github/Alex-Zeo/audio_downloader/blob/main/Audio_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Audio Downloader - Public Media - X Spaces, Youtube, Apple Podcasts, Spotify
*   Does: downloads public videos, podcasts, playlists, or channels/authors
*   Input: URL (non DRMM protected only for academic purposes or for personal transcription)
  
Dependencies:
  - yt_dlp, whisper-openai, tqdm, pydub, ffmpeg
  - For Google Colab: google.colab.driv

In [1]:
# Install required packages (if running in Colab):
# ---------------------------------------------------------
!pip install --quiet --upgrade yt_dlp tqdm pydub
!apt-get update && apt-get install -y ffmpeg

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 10.2 kB in 3s (3,077 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
%%writefile audio_downloader.py
#!/usr/bin/env python3
"""
Audio Downloader
---------------------------------------
This script downloads high-quality audio tracks (in WAV) from multiple YouTube, Twitter Spaces, and Spotify Podcast URLs
using yt_dlp. Designed for Google Colab environment.

Usage (Colab Example):
  !python audio_downloader.py --urls "https://youtube.com/watch?v=..." "https://x.com/i/spaces/..." ""

The script will download high-quality WAV files directly to your local downloads folder.
"""

import sys
import os
import time
import logging
import logging.config
import argparse
from tqdm import tqdm
import yt_dlp
from yt_dlp.utils import DownloadError
import shutil
import tempfile
import glob

# Try to import Colab-specific modules, but continue if not available
try:
    from google.colab import files
    from IPython.display import display, HTML
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# --------------------------------------------------------------------------
# 1) LOGGING CONFIG
# --------------------------------------------------------------------------
logging_config = {
    'version': 1,
    'disable_existing_loggers': False,
    'formatters': {
        'json': {
            'format': (
                '{"time": "%(asctime)s", "level": "%(levelname)s", '
                '"module": "%(module)s", "lineno": %(lineno)d, '
                '"message": "%(message)s"}'
            )
        }
    },
    'handlers': {
        'console': {
            'class': 'logging.StreamHandler',
            'formatter': 'json'
        },
    },
    'root': {
        'handlers': ['console'],
        'level': 'INFO',
    },
}
logging.config.dictConfig(logging_config)
logger = logging.getLogger(__name__)

# --------------------------------------------------------------------------
# 2) PROGRESS BAR HOOK
# --------------------------------------------------------------------------
#   This function hooks into yt_dlp's progress reporting.
# --------------------------------------------------------------------------
pbar = None
def progress_hook(d):
    global pbar
    if d.get('status') == 'downloading':
        if pbar is None:
            total_bytes = d.get('total_bytes') or d.get('total_bytes_estimate') or 0
            if total_bytes == 0:
                total_bytes = None  # TQDM can handle None as unknown total
            pbar = tqdm(total=total_bytes, unit='B', unit_scale=True, desc="Downloading")
        downloaded_bytes = d.get('downloaded_bytes', 0)
        pbar.update(downloaded_bytes - pbar.n)
    elif d.get('status') == 'finished':
        if pbar:
            pbar.close()
            pbar = None

# --------------------------------------------------------------------------
# 3) MAIN DOWNLOAD FUNCTION
# --------------------------------------------------------------------------
def download_audio(url: str, output_dir: str, keep_files: bool = False) -> str:
    """
    Download audio from the given URL and save as high-quality WAV.
    Returns the path to the downloaded file if successful, None otherwise.

    :param url: The YouTube (or supported site) URL.
    :param output_dir: The directory to save the WAV file.
    :param keep_files: Whether to keep original files after extraction.
    """
    logger.info(f"Preparing to download audio from: {url}")

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Extract clean filename from URL
    # Remove protocol (http/https)
    if "://" in url:
        clean_url = url.split('://')[-1]
    else:
        clean_url = url

    # Keep only domain and video ID part
    # For YouTube, this preserves the 'youtube.com_watch?v=abc123' format
    if '?' in clean_url:
        base_url = clean_url.split('?')[0]
        params = clean_url.split('?')[1].split('&')

        # Keep only essential parameters (like v= for YouTube)
        essential_params = []
        for param in params:
            # For YouTube, keep the video ID parameter
            if param.startswith('v='):
                essential_params.append(param)
            # For other sites with ID parameters, you can add conditions here

        if essential_params:
            clean_url = f"{base_url}?{'&'.join(essential_params)}"
        else:
            clean_url = base_url

    # Remove trailing slashes
    clean_url = clean_url.rstrip('/')

    # Replace / and other problematic characters with underscores
    clean_url = clean_url.replace('/', '_')

    # Limit length if needed
    if len(clean_url) > 100:
        clean_url = clean_url[:100]

    output_path = os.path.join(output_dir, f"{clean_url}.wav")

    # Remove existing partial file if present
    if os.path.exists(output_path):
        logger.warning(f"Removing existing file before download: {output_path}")
        os.remove(output_path)

    # Get base name without extension for yt_dlp
    base_without_ext = os.path.splitext(output_path)[0]

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': base_without_ext,
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '0',  # Highest quality for WAV (lossless)
        }],
        'retries': 3,
        'progress_hooks': [progress_hook],
        'keepvideo': keep_files,  # Keep original files if requested
    }

    max_retries = 3
    for attempt in range(max_retries):
        logger.info(f"Download attempt {attempt+1}/{max_retries}")
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])
            logger.info("Download finished.")
            break
        except DownloadError as err:
            logger.warning(f"Download error on attempt {attempt+1}: {err}")
            if attempt == max_retries - 1:
                logger.error("Exceeded maximum retries. Aborting download.")
                return None
            time.sleep(2)  # brief pause before retry

    final_wav = base_without_ext + ".wav"
    if os.path.exists(final_wav) and os.path.getsize(final_wav) > 0:
        logger.info(f"Success: Downloaded file => {final_wav}")
        return final_wav
    else:
        logger.error(f"Failed to produce WAV file => {final_wav}")
        return None

# --------------------------------------------------------------------------
# 4) CLI / MAIN
# --------------------------------------------------------------------------
def parse_args():
    parser = argparse.ArgumentParser(
        description="Download high-quality WAV audio from multiple YouTube URLs."
    )
    parser.add_argument("--urls", required=True, nargs='+', help="List of URLs to download.")
    parser.add_argument("--output_dir", default="./audio_downloads",
                      help="Local directory to save WAV files when not in Colab.")
    parser.add_argument("-k", "--keep", action="store_true",
                      help="Keep original files and save in current directory instead of temporary folder.")
    return parser.parse_args()

def download_to_local(file_path: str) -> bool:
    """Download a file to the user's local machine from Colab."""
    if not os.path.exists(file_path) or os.path.getsize(file_path) == 0:
        logger.error(f"File doesn't exist or is empty: {file_path}")
        return False

    filename = os.path.basename(file_path)
    try:
        if IS_COLAB:
            try:
                # Using Colab's files.download
                logger.info(f"Downloading {filename} to your local downloads folder using Colab API")
                files.download(file_path)
                return True
            except Exception as e:
                logger.warning(f"Colab download API failed: {e}. File is still available at {file_path}")
                # Display a download link as fallback
                display(HTML(f'<a href="./files/{filename}" download="{filename}">Download {filename}</a>'))
                return True
        else:
            # We're already in a local environment
            logger.info(f"File saved locally at: {file_path}")
            return True
    except Exception as e:
        logger.error(f"Failed to download {file_path}: {e}")
        return False

def generate_download_links(files):
    """Generate HTML download links for files in Colab."""
    if not IS_COLAB or not files:
        return

    logger.info("Generating download links for the files:")
    html = "<h3>Download your audio files:</h3><ul>"
    for file_path in files:
        filename = os.path.basename(file_path)
        html += f'<li><a href="./files/{filename}" download="{filename}">{filename}</a></li>'
    html += "</ul>"
    display(HTML(html))

def main():
    args = parse_args()

    # Determine the output directory based on whether we're keeping files
    if IS_COLAB and args.keep:
        # Use current directory when keeping files in Colab
        output_dir = "."
        logger.info(f"Running in Google Colab environment with --keep flag. Saving to current directory.")
        use_temp_dir = False
    elif IS_COLAB and not args.keep:
        # Use temp directory on Colab when not keeping files
        logger.info("Running in Google Colab environment")
        use_temp_dir = True
    else:
        # Use specified output directory on local machine
        logger.info("Running in local environment")
        use_temp_dir = False
        output_dir = args.output_dir
        os.makedirs(output_dir, exist_ok=True)

    success_count = 0
    failed_urls = []
    downloaded_files = []

    # Context manager for temp directory if needed
    if use_temp_dir:
        with tempfile.TemporaryDirectory() as temp_dir:
            logger.info(f"Created temporary directory for downloads: {temp_dir}")
            output_dir = temp_dir
            # Download process
            for url in args.urls:
                output_file = download_audio(url, output_dir, args.keep)
                if output_file:
                    downloaded_files.append(output_file)
                    success_count += 1
                else:
                    failed_urls.append(url)

            # For Colab, try to download via files.download API
            logger.info(f"Download summary: {success_count}/{len(args.urls)} successful downloads")
            if downloaded_files:
                logger.info(f"Downloading {len(downloaded_files)} WAV files to your local computer...")
                for file_path in downloaded_files:
                    download_to_local(file_path)
    else:
        # Using permanent directory (current dir for Colab --keep, or specified dir for local)
        logger.info(f"Saving files to directory: {output_dir}")
        # Download process
        for url in args.urls:
            output_file = download_audio(url, output_dir, args.keep)
            if output_file:
                downloaded_files.append(output_file)
                success_count += 1
            else:
                failed_urls.append(url)

        logger.info(f"Download summary: {success_count}/{len(args.urls)} successful downloads")
        if failed_urls:
            logger.warning(f"Failed URLs: {failed_urls}")

        if downloaded_files and IS_COLAB:
            logger.info(f"Files saved to current directory. Generating download links...")
            # Generate clickable download links for all WAV files
            generate_download_links(downloaded_files)
        elif downloaded_files:
            logger.info(f"Files saved to: {output_dir}")
        else:
            logger.warning("No files were successfully downloaded.")

if __name__ == "__main__":
    main()

Overwriting audio_downloader.py


In [10]:
!python audio_downloader.py -k --urls "https://youtube.com/watch?v=dQw4w9WgXcQ" "https://x.com/TuckerCarlson/status/1894809980940730574"

{"time": "2025-03-26 18:19:44,266", "level": "INFO", "module": "audio_downloader", "lineno": 245, "message": "Running in Google Colab environment with --keep flag. Saving to current directory."}
{"time": "2025-03-26 18:19:44,266", "level": "INFO", "module": "audio_downloader", "lineno": 284, "message": "Saving files to directory: ."}
{"time": "2025-03-26 18:19:44,266", "level": "INFO", "module": "audio_downloader", "lineno": 97, "message": "Preparing to download audio from: https://youtube.com/watch?v=dQw4w9WgXcQ"}
{"time": "2025-03-26 18:19:44,267", "level": "INFO", "module": "audio_downloader", "lineno": 163, "message": "Download attempt 1/3"}
[youtube] Extracting URL: https://youtube.com/watch?v=dQw4w9WgXcQ
[youtube] dQw4w9WgXcQ: Downloading webpage
[youtube] dQw4w9WgXcQ: Downloading tv client config
[youtube] dQw4w9WgXcQ: Downloading player 20830619
[youtube] dQw4w9WgXcQ: Downloading tv player API JSON
[youtube] dQw4w9WgXcQ: Downloading ios player API JSON
[youtube] dQw4w9WgXcQ: Do